In [1]:
import glob
from pathlib import Path
import pandas as pd
import polars as pl

import aw
import job_search.jobs as jb
import job_search.config as conf
import job_search.scrape as sc
from job_search.config import P_RAW, P_INTERIM
from job_search.utils import now, reload
from job_search.scrape import DA_SF, DS_SF, DA_HEALTH, HEALTH

_now = now(time=False, days=0)
# _now = '2026-04-20'
P_raw_date = P_RAW / _now.replace('-', '/')
P_interim_date = P_INTERIM / _now.replace('-', '/')

In [409]:
reload(sc)

In [414]:
glob_prefix = '../data/interim/2026/05/**'
health_df = sc.load_json_gz(glob_health := f'{glob_prefix}/{HEALTH}/*.json.gz', metadata=True)
ds_sf_df = sc.load_json_gz(glob_ds_sf := f'{glob_prefix}/{DS_SF}/*.json.gz', metadata=True)
da_health_df = sc.load_json_gz(glob_da_health := f'{glob_prefix}/{DA_HEALTH}/*.json.gz', metadata=True)
da_sf_df = sc.load_json_gz(glob_da_sf := f'{glob_prefix}/{DA_SF}/*.json.gz', metadata=True)

In [411]:
health_df.shape[0], ds_sf_df.shape[0], da_health_df.shape[0], da_sf_df.shape[0]

(8809, 8869, 4227, 6683)

In [412]:
from markdownify import markdownify as md

def _process_df(jobs_df):
    jobs_df['_url'] = jb.VIEW_JOB_HTTPS + jobs_df['requisition_id']
    jobs_df['_hash'] = jobs_df['requisition_id']
    jobs_df['_md'] = jobs_df['description']
    jobs_df = (jb.load_jobs(jobs_df)
        .sort_values('estimated_publish_date', ascending=False)
        .dropna(subset='_hash')
        .drop_duplicates(subset='_hash')
        .reset_index(drop=True)
    )#[[*jb.COLS, 'location_latitudes', 'location_longitudes']]
    jobs_df['_md'] = jobs_df['description'].fillna('').map(md)
    return jobs_df

In [428]:
health_df.dropna(subset='requisition_id')[health_df.dropna(subset='requisition_id')['description'].isna()]['st_mtime'].min()

Timestamp('2026-05-16 02:30:17.313909531')

In [421]:
health_df[health_df['description'].isna()]['st_mtime'].min()

Timestamp('2026-05-03 00:33:14.987561941')

In [330]:
health_parquet_df = _process_df(health_df)
health_parquet_df.to_parquet(f'../data/cache/{HEALTH}.parquet')

In [365]:
ds_sf_parquet_df = _process_df(ds_sf_df)
ds_sf_parquet_df.to_parquet(f'../data/cache/{DS_SF}.parquet')

In [361]:
da_sf_parquet_df = _process_df(da_sf_df)
da_sf_parquet_df.to_parquet(f'../data/cache/{DA_SF}.parquet')

In [362]:
da_health_parquet_df = _process_df(da_health_df)
da_health_parquet_df.to_parquet(f'../data/cache/{DA_HEALTH}.parquet')

In [441]:
all_df = (pd.concat([
    health_df,
    ds_sf_df,
    da_sf_df,
    da_health_df,
]).sort_values(['st_mtime', 'estimated_publish_date'], ascending=True)
    .dropna(subset='requisition_id')
    .drop_duplicates(subset='requisition_id')
    .reset_index(drop=True)
)
# all_df['st_mtime'].mean()
all_df.drop_duplicates(subset='requisition_id')
all_df

,st_mtime,st_size,requisition_id,job_id,board_token,source,apply_url,collapse_key,is_expired,title,...,company_tagline,company_organization_type,company_latest_funding_investors),company_latest_funding_type,company_latest_funding_year,company_latest_funding_amount,company_stock_exchange,company_stock_symbol,location_longitudes,location_latitudes
0,2026-05-03 00:31:34.858393908,350676,309eiqubmllkwa14,grnhse___elementbiosciences___5728046004,elementbiosciences,grnhse,https://job-boards.greenhouse.io/elementbiosci...,be4d74a6e10c2c0da07a720d20c754918e33b310662f39...,False,Sr Embedded Systems Engineer,...,Develops modular DNA sequencing platforms for ...,Private,[Wellington Management],Series D,2024.0,277000000.0,None,None,[-117.1611],[32.7157]
1,2026-05-03 00:31:34.858393908,350676,19la7bfe9xlznd5t,grnhse___flatironhealth___7511628,flatironhealth,grnhse,https://flatiron.com/careers/open-positions/jo...,455c79b0c777c9102be4660c38b8cfd82b2fb18bd5138c...,False,Senior Data Engineer,...,Provides oncology software and clinical data f...,Public,None,None,NaN,NaN,SIX Swiss Exchange,ROG,[-74.006],[40.7128]
2,2026-05-03 00:31:34.858393908,350676,llorpv5d02n7r8y8,utipro___wom1000whf___2f4bcf30-0e2d-4c6d-82e3-...,wom1000whf_2f4bcf30-0e2d-4c6d-82e3-d7e885e672ea,ultipro,https://recruiting2.ultipro.com/wom1000whf/Job...,4ab38394944d9df62aeab2c7c27ec08716ecf9e6faea3f...,False,BI Developer II - Information Systems,...,Specialty hospital providing healthcare for wo...,Non-Profit,None,None,NaN,NaN,None,None,[-91.037856],[30.3855]
3,2026-05-03 00:31:34.858393908,350676,m6cg3i25t85v4noc,grnhse___flatironhealth___7681161,flatironhealth,grnhse,https://flatiron.com/careers/open-positions/jo...,455c79b0c777c9102be4660c38b8cfd82b2fb18bd5138c...,False,Software Engineer (Contractor),...,Provides oncology software and clinical data f...,Public,None,None,NaN,NaN,SIX Swiss Exchange,ROG,[-74.006],[40.7128]
4,2026-05-03 00:31:34.858393908,350676,96ustvxzv9cn7cpc,grnhse___zocdoc___7736528,zocdoc,grnhse,https://job-boards.greenhouse.io/zocdoc/jobs/7...,d4dc3fe353aa194dbcd683834bb10dbadce6fe37300c50...,False,Senior Site Reliability Engineer,...,Online platform for searching and booking heal...,Private,[Francisco Partners],Series E,2021.0,150000000.0,None,None,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,2026-05-29 18:28:53.492630243,144490,nyheyceukdtek57s,ashby___brightwheel___588834a1-4c36-4bd0-92f5-...,brightwheel,ashby,https://jobs.ashbyhq.com/brightwheel/588834a1-...,faccdecca48065be66243b21ae9999a105a0261f8e4440...,False,AI Campaign Operations Specialist,...,Provides an all-in-one management platform for...,Private,"[Addition, Bessemer Venture Partners]",Series C,2022.0,50000000.0,None,None,[],[]
11110,2026-05-29 18:28:53.492630243,144490,z9h2jdub2jg3rq7p,lever___get-vocal-pbc___552b5a34-a0e0-4e2a-b0a...,get-vocal-pbc,lever,https://jobs.lever.co/get-vocal-pbc/552b5a34-a...,14fe011c61efac4d12d524df589afc33a3540d40549bb6...,False,Research Analyst,...,Strategic influencer marketing for social impa...,Private,[Higher Ground Labs],Angel,2023.0,500000.0,None,None,[],[]
11111,2026-05-29 18:28:53.492630243,144490,lirwzbs8jo4edw7n,ashby___jellyfish___a7aa0e73-e3f9-4bcd-9886-dc...,jellyfish,ashby,https://jobs.ashbyhq.com/jellyfish/a7aa0e73-e3...,f35f2c8e81b4a8900adf995b0d6ec09dbfa761efc317a3...,False,Staff Data Architect,...,Analytics platform for tracking software engin...,Private,"[Accel, Insight Partners, Tiger Global]",Series C,2022.0,71000000.0,None,None,[],[]
11112,2026-05-29 18:28:53.492630243,144490,fe0iv95uxwfdt1hr,grnhse___birdygrey___5145569007,birdygrey,grnhse,https://job-boards.greenhouse.io/birdygrey/job...,896e2ea101e9b93bd1548a0005b858ed2efeb6eb0bde19...,False,Senior Strategy Associate,...,Direct-to-consumer brand selling affordable br...,Private,"[Bling Capital, BAM Ventures, Labora Group]",Seed,2019.0,2500000.0,None,None,[],[]


In [429]:
all_parquet_df = (pd.concat([
        health_parquet_df,
        ds_sf_parquet_df,
        da_sf_parquet_df,
        da_health_parquet_df,
    ]).sort_values('estimated_publish_date', ascending=False)
        .dropna(subset='_hash')
        .drop_duplicates(subset='_hash')
        .reset_index(drop=True)
    )
all_df.to_parquet(f'../data/cache/ALL.parquet')

## Save JSON descriptions

In [465]:
# health_df[health_df['description'].isna()].sort_values('st_mtime', ascending=False)[lambda x: x['company_name'].fillna('').str.contains('Tempus')]

In [460]:
health_df[health_df['description'].isna()].sort_values('st_mtime', ascending=False)
# all_df[all_df['description'].isna()].sort_values('st_mtime', ascending=False)

,st_mtime,st_size,requisition_id,job_id,board_token,source,apply_url,collapse_key,is_expired,title,...,company_tagline,company_organization_type,company_latest_funding_investors),company_latest_funding_type,company_latest_funding_year,company_latest_funding_amount,company_stock_exchange,company_stock_symbol,location_longitudes,location_latitudes
8808,2026-05-29 18:25:41.462252378,146092,ln9dcblqjxza1q7e,dayforce___nant___candidateportal___1033,nant___candidateportal___en-us,dayforce,https://jobs.dayforcehcm.com/en-US/nant/candid...,ee1fa4206e4ea3537fdd8f49e7807ef726b875041ff585...,False,Senior Systems Analyst | Remote | NantHealth,...,Developing personalized medical treatments and...,Private,[Allscripts],Corporate Round,2015.0,200000000.0,None,None,[-98.4945922],[29.4251905]
8751,2026-05-29 18:25:41.462252378,146092,511cnhej7r441szd,csod___uisystemoffice___2984,uisystemoffice___5,csod,https://uisystemoffice.csod.com/ux/ats/careers...,e4830b78258eb92a81f2c459ee0991ed316e8f9467c969...,False,Data Analyst - Remote,...,"Provides higher education, research, and healt...",Government,None,None,NaN,NaN,None,None,[-88.20727],[40.1105881]
8742,2026-05-29 18:25:41.462252378,146092,wt7m18kf34cev5vq,workday___streamlinehealthcare-wd501-streamlin...,streamlinehealthcare-wd501-streamline_healthca...,workday,https://streamlinehealthcare.wd501.myworkdayjo...,17b7bad6abcdfdbd39ffa65672657d1cf3140b30a548cd...,False,Software Engineer,...,Provides electronic health record software for...,Private,[],Private Equity,2023.0,15400000.0,None,None,[],[]
8743,2026-05-29 18:25:41.462252378,146092,lqlwhzoslqqa38tq,workday___streamlinehealthcare-wd501-streamlin...,streamlinehealthcare-wd501-streamline_healthca...,workday,https://streamlinehealthcare.wd501.myworkdayjo...,17b7bad6abcdfdbd39ffa65672657d1cf3140b30a548cd...,False,Software Engineer,...,Provides electronic health record software for...,Private,[],Private Equity,2023.0,15400000.0,None,None,[],[]
8744,2026-05-29 18:25:41.462252378,146092,3vovhx5k0m2unuzi,workday___streamlinehealthcare-wd501-streamlin...,streamlinehealthcare-wd501-streamline_healthca...,workday,https://streamlinehealthcare.wd501.myworkdayjo...,17b7bad6abcdfdbd39ffa65672657d1cf3140b30a548cd...,False,Lead AI Software Engineer,...,Provides electronic health record software for...,Private,[],Private Equity,2023.0,15400000.0,None,None,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5515,2026-05-16 02:30:17.313909531,157299,fwz5q453mt0quu54,smartrecruiters___abbvie___a3a02346-1f0a-4459-...,abbvie,smartrecruiters,https://jobs.smartrecruiters.com/AbbVie/374399...,6f4d122a6f9c4b5685fa20dabcef73ed2bcc0f8b9c0d04...,False,Data Science Program Lead I,...,Develops and sells innovative pharmaceutical a...,Public,None,None,NaN,NaN,NYSE,ABBV,[-74.3893296],[40.7881053]
5516,2026-05-16 02:30:17.313909531,157299,862zn43jecjm9f00,utipro___vor1001vori___e5f59222-eab9-4772-8e33...,vor1001vori_e5f59222-eab9-4772-8e33-f436c4f4a1f6,ultipro,https://recruiting.ultipro.com/vor1001vori/Job...,235ccf0eb154cbe9370fede0ccf2bb41a564bf25e9f8c9...,False,Senior Data Integration Engineer (100% Remote),...,Virtual-first medical practice specializing in...,Private,[New Enterprise Associates],Series B,2025.0,53000000.0,None,None,[-86.76582],[36.1059]
5438,2026-05-16 02:30:17.313909531,157299,7y3lzczsc9ra8ywu,oraclecloud___ecvz.fa.us2___63752,ecvz.fa.us2,oraclecloud,https://ecvz.fa.us2.oraclecloud.com/hcmUI/Cand...,ccb17ca98ec57867d3a343a43321976b861e7ce3e25317...,False,"Clinical Laboratory Scientist, Full Time, Nigh...",...,Faith-based healthcare provider offering hospi...,Non-Profit,None,None,NaN,NaN,None,None,[-123.80592],[39.44672]
4150,2026-05-08 21:41:37.239470005,1258,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,...,None,NaN,None,None,NaN,NaN,None,None,[],[]


In [654]:
len([p for p in (jb.P_CACHE / 'json').iterdir()])

2402

In [470]:
proxy = False
driver = sc.init_driver(proxy=proxy, headless=False)

errors = []
for _hash in tqdm(all_df[all_df['description'].isna()].sort_values('st_mtime', ascending=False)['requisition_id']):
    # P_url = jb.P_CACHE / 'json' / f'2xztjhutpo56dvg9.json.gz'
    P_url = jb.P_CACHE / 'url' / f'{_hash}.html'
    P_json = jb.P_CACHE / 'json' / f'{_hash}.json.gz'
    if P_url.exists():
        continue

    # url = jb.VIEW_JOB_HTTPS + _hash
    url = jb.JOB_HTTPS + _hash
    print(url)

    # url_get_content = sc.selenium_get(url, driver=driver)
    try:
        url_get_content = sc.requests_get(url)
    except Exception:
        url_get_content = sc.selenium_get(url, driver=driver)

    root = lxml.html.fromstring(url_get_content)
    _next_data_list = root.xpath("//script[@id='__NEXT_DATA__']")
    if len(_next_data_list) == 0:
        jobs_dict = {}
        errors.append(url)
        print(url)
        continue
    else:
        _next_data = root.xpath("//script[@id='__NEXT_DATA__']")[0]
        jobs_dict = json.loads(_next_data.text_content())
        jobs_dict = jobs_dict.get('props', jobs_dict)

    sc.write_data(P_json, jobs_dict)
    with open(P_url, "w", encoding="utf-8") as f:
        f.write(url_get_content)
    sc.sleep(0.2, 0.5)

  0%|          | 0/2404 [00:00<?, ?it/s]

https://hiring.cafe/job/e97ozk2yyfmxqz4f
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\e97ozk2yyfmxqz4f.json.gz


  0%|          | 1/2404 [00:01<53:33,  1.34s/it]

https://hiring.cafe/job/d3ddzqwxd22qwbjk
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\d3ddzqwxd22qwbjk.json.gz


  0%|          | 2/2404 [00:02<52:23,  1.31s/it]

...

100%|█████████▉| 2402/2404 [49:59<00:02,  1.26s/it]

https://hiring.cafe/job/fgzu1yrbdug37j95
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\fgzu1yrbdug37j95.json.gz


100%|█████████▉| 2403/2404 [50:00<00:01,  1.22s/it]

https://hiring.cafe/job/lfa4iw7p25ejvdu4
Writing: C:\Users\Alex\Dev\job-search\data\cache\json\lfa4iw7p25ejvdu4.json.gz


100%|██████████| 2404/2404 [50:01<00:00,  1.25s/it]


In [371]:
url = 'https://hiring.cafe/job/2xztjhutpo56dvg9'

In [382]:
import json
import lxml.html

url_get_content = sc.requests_get(url)
root = lxml.html.fromstring(url_get_content)
_next_data_list = root.xpath("//script[@id='__NEXT_DATA__']")
if len(_next_data_list) == 0:
    jobs_dict = {}
else:
    _next_data = root.xpath("//script[@id='__NEXT_DATA__']")[0]
    jobs_dict = json.loads(_next_data.text_content())
    jobs_dict = jobs_dict.get('props', jobs_dict)

In [395]:
(P_json := jb.P_CACHE / 'json' / '2xztjhutpo56dvg9.json.gz').exists()

False

In [508]:
# glob_str = P_CACHE / 'jso
glob_str = '../data/cache/json/*.json.gz'
json_gz_paths = sorted([Path(path) for path in glob.glob(glob_str, recursive=True)])
# pl_df = pl.read_ndjson(json_gz_paths)
json_gz_list = []
for P_json_gz in tqdm(json_gz_paths):
    _json_gz_df = pd.read_json(P_json_gz, lines=True, compression='gzip')
    json_gz_list.append(_json_gz_df)
pl_df = pd.concat(json_gz_list)
_pageProps = pl_df['pageProps']

100%|██████████| 2402/2402 [00:19<00:00, 125.32it/s]


In [510]:
json_gz_df = pd.DataFrame({
    'hits': _pageProps,
})
json_gz_df['st_mtime'] = pd.to_datetime([p.stat().st_mtime for p in json_gz_paths], unit='s')
json_gz_df['st_size'] = [p.stat().st_size for p in json_gz_paths]
json_gz_df = json_gz_df[json_gz_df['hits'].str.len() == 1].reset_index(drop=True)

In [ ]:
df['hits']

,hits,st_mtime,st_size
0,{'job': {'id': 'ashby___zodl___cee86612-7f08-4...,2026-05-30 01:37:59.529092311,5923
1,{'job': {'id': 'grnhse___folxhealth___51379940...,2026-05-30 02:03:15.041346312,10616
2,{'job': {'id': 'utipro___bet1003beth___a446e30...,2026-05-30 01:31:02.924612761,4662
3,{'job': {'id': 'workday___medtronic-wd1-medtro...,2026-05-30 01:58:54.926348686,7709
4,{'job': {'id': 'workday___uw-wd5-uwhires___dat...,2026-05-30 01:20:11.853490114,7555
...,...,...,...
2392,"{'job': {'id': 'adhoc___amazon___10421923', 'b...",2026-05-30 01:52:35.829968691,4580
2393,{'job': {'id': 'grnhse___companycam___77484430...,2026-05-30 01:18:07.847062111,6029
2394,{'job': {'id': 'ttcportals___syneoshealthcaree...,2026-05-30 01:24:07.176661015,5840
2395,{'job': {'id': 'workday___roche-wd3-roche-ext_...,2026-05-30 01:35:51.281837940,5302


In [ ]:
df = json_gz_df#.explode('hits')
job = df['hits'].str['job']
job_info = job.str['job_information']
v5_processed = job.str['v5_processed_job_data']
v5_company_data = job.str['v5_processed_company_data'].apply(lambda x: x if isinstance(x, dict) else {})
company_data = job.str['enriched_company_data']

df['requisition_id'] = job.str['requisition_id']
df['job_id'] = job.str['id']
df['board_token'] = job.str['board_token'].astype(str)
df['source'] = job.str['source']
df['apply_url'] = job.str['apply_url']
df['collapse_key'] = job.str['collapse_key']
df['is_expired'] = job.str['is_expired']

# Job information
df['title'] = job_info.str['title']
df['job_title_raw'] = job_info.str['job_title_raw']
df['description'] = job_info.str['description']
df['core_job_title'] = v5_processed.str['core_job_title']
df['requirements_summary'] = v5_processed.str['requirements_summary']

# Structured data
df['technical_tools'] = v5_processed.str['technical_tools']
df['licenses_certifications'] = v5_processed.str['licenses_or_certifications']
df['role_activities'] = v5_processed.str['role_activities']
df['language_requirements'] = v5_processed.str['language_requirements']

# Degree requirements
df['associates_degree_requirement'] = v5_processed.str['associates_degree_requirement']
df['associates_degree_fields'] = v5_processed.str['associates_degree_fields_of_study']
df['bachelors_degree_requirement'] = v5_processed.str['bachelors_degree_requirement']
df['bachelors_degree_fields'] = v5_processed.str['bachelors_degree_fields_of_study']
df['masters_degree_requirement'] = v5_processed.str['masters_degree_requirement']
df['masters_degree_fields'] = v5_processed.str['masters_degree_fields_of_study']
df['doctorate_degree_requirement'] = v5_processed.str['doctorate_degree_requirement']
df['doctorate_degree_fields'] = v5_processed.str['doctorate_degree_fields_of_study']
df['is_high_school_required'] = v5_processed.str['is_high_school_required']

# Experience requirements
df['min_industry_role_yoe'] = v5_processed.str['min_industry_and_role_yoe']
df['is_min_industry_role_yoe_not_mentioned'] = v5_processed.str['is_min_industry_and_role_yoe_not_mentioned']
df['min_management_leadership_yoe'] = v5_processed.str['min_management_and_leadership_yoe']
df['is_min_management_leadership_yoe_not_mentioned'] = v5_processed.str['is_min_management_and_leadership_yoe_not_mentioned']

# Role details
df['job_category'] = v5_processed.str['job_category']
df['commitment'] = v5_processed.str['commitment']
df['role_type'] = v5_processed.str['role_type']
df['seniority_level'] = v5_processed.str['seniority_level']

# Workplace
df['workplace_type'] = v5_processed.str['workplace_type']
df['workplace_physical_environment'] = v5_processed.str['workplace_physical_environment']
df['formatted_workplace_location'] = v5_processed.str['formatted_workplace_location']
df['is_workplace_worldwide_ok'] = v5_processed.str['is_workplace_worldwide_ok']
df['workplace_cities'] = v5_processed.str['workplace_cities']
df['workplace_counties'] = v5_processed.str['workplace_counties']
df['workplace_states'] = v5_processed.str['workplace_states']
df['workplace_countries'] = v5_processed.str['workplace_countries']
df['workplace_continents'] = v5_processed.str['workplace_continents']
df['boundless_workplace_states'] = v5_processed.str['boundless_workplace_states']
df['boundless_workplace_countries'] = v5_processed.str['boundless_workplace_countries']
df['boundless_workplace_continents'] = v5_processed.str['boundless_workplace_continents']
df['number_of_workplace_cities'] = v5_processed.str['number_of_workplace_cities']
df['number_of_workplace_counties'] = v5_processed.str['number_of_workplace_counties']
df['number_of_workplace_states'] = v5_processed.str['number_of_workplace_states']
df['number_of_workplace_countries'] = v5_processed.str['number_of_workplace_countries']
df['number_of_workplace_continents'] = v5_processed.str['number_of_workplace_continents']

# Work conditions
df['oral_communication_level'] = v5_processed.str['oral_communication_level']
df['physical_labor_intensity'] = v5_processed.str['physical_labor_intensity']
df['physical_position'] = v5_processed.str['physical_position']
df['computer_usage'] = v5_processed.str['computer_usage']
df['cognitive_demand'] = v5_processed.str['cognitive_demand']
df['air_travel_requirement'] = v5_processed.str['air_travel_requirement']
df['land_travel_requirement'] = v5_processed.str['land_travel_requirement']
df['morning_shift_work'] = v5_processed.str['morning_shift_work']
df['evening_shift_work'] = v5_processed.str['evening_shift_work']
df['overnight_work'] = v5_processed.str['overnight_work']
df['on_call_requirement'] = v5_processed.str['on_call_requirement']
df['weekend_availability_required'] = v5_processed.str['weekend_availability_required']
df['holiday_availability_required'] = v5_processed.str['holiday_availability_required']
df['overtime_required'] = v5_processed.str['overtime_required']

# Compensation
df['yearly_min_compensation'] = v5_processed.str['yearly_min_compensation']
df['yearly_max_compensation'] = v5_processed.str['yearly_max_compensation']
df['monthly_min_compensation'] = v5_processed.str['monthly_min_compensation']
df['monthly_max_compensation'] = v5_processed.str['monthly_max_compensation']
df['weekly_min_compensation'] = v5_processed.str['weekly_min_compensation']
df['weekly_max_compensation'] = v5_processed.str['weekly_max_compensation']
df['hourly_min_compensation'] = v5_processed.str['hourly_min_compensation']
df['hourly_max_compensation'] = v5_processed.str['hourly_max_compensation']
df['biweekly_min_compensation'] = v5_processed.str['bi-weekly_min_compensation']
df['biweekly_max_compensation'] = v5_processed.str['bi-weekly_max_compensation']
df['daily_min_compensation'] = v5_processed.str['daily_min_compensation']
df['daily_max_compensation'] = v5_processed.str['daily_max_compensation']
df['is_compensation_transparent'] = v5_processed.str['is_compensation_transparent']
df['listed_compensation_currency'] = v5_processed.str['listed_compensation_currency']
df['listed_compensation_frequency'] = v5_processed.str['listed_compensation_frequency']

# Benefits
df['four_oh_one_k_matching'] = v5_processed.str['401k_matching']
df['generous_paid_time_off'] = v5_processed.str['generous_paid_time_off']
df['four_day_work_week'] = v5_processed.str['four_day_work_week']
df['tuition_reimbursement'] = v5_processed.str['tuition_reimbursement']
df['retirement_plan'] = v5_processed.str['retirement_plan']
df['generous_parental_leave'] = v5_processed.str['generous_parental_leave']
df['fair_chance'] = v5_processed.str['fair_chance']
df['visa_sponsorship'] = v5_processed.str['visa_sponsorship']
df['relocation_assistance'] = v5_processed.str['relocation_assistance']
df['military_veterans'] = v5_processed.str['military_veterans']

# Other
df['security_clearance'] = v5_processed.str['security_clearance']
df['is_driver_license_required'] = v5_processed.str['is_driver_license_required']
df['position_employer_type'] = v5_processed.str['position_employer_type']
df['company_sector_and_industry'] = v5_processed.str['company_sector_and_industry']

# Dates
_estimated_publish_date = v5_processed.str['estimated_publish_date']
# if isinstance(_estimated_publish_date, int):
#     _estimated_publish_date = datetime.fromtimestamp(_estimated_publish_date / 1000)
df['estimated_publish_date'] = _estimated_publish_date
df['estimated_publish_date_millis'] = v5_processed.str['estimated_publish_date_millis']

_v5_is_public = v5_company_data.str['is_public']
# _v5_org_type = {True: 'Public', False: 'Private'}.get(_v5_is_public)
# if v5_company_data.get('is_non_profit'):
#     _v5_org_type = 'Non-Profit'
_v5_org_type = _v5_is_public.map({True: 'Public', False: 'Private'})
_v5_org_type.loc[v5_company_data.str['is_non_profit'].astype(bool)] = 'Non-Profit'

df = pd.concat([df,
    # User interactions
    job_info.str['hiddenFromUsers'].rename('hiddenFromUsers'),
    job_info.str['viewedByUsers'].rename('viewedByUsers'),
    job_info.str['applied_from_users'].rename('applied_from_users'),

    # Enriched Company data (embedded)
    company_data.str['enriched_at'].rename('company_enriched_at'),
    company_data.str['status'].rename('company_status'),
    company_data.str['name'].fillna(v5_processed.str['company_name']).fillna(v5_company_data.str['name']).rename('company_name'),
    company_data.str['homepage_uri'].fillna(v5_processed.str['company_website']).fillna(v5_company_data.str['website']).rename('company_homepage_uri'),
    company_data.str['hq_country'].fillna(v5_company_data.str['headquarters_country']).rename('company_hq_country'),
    company_data.str['parent_company'].fillna(v5_company_data.str['parent_company']).rename('company_parent_company'),
    company_data.str['subsidiaries'].fillna(v5_company_data.str['subsidiaries']).rename('company_subsidiaries'),
    company_data.str['industries'].fillna(v5_company_data.str['industries']).rename('company_industries'),
    company_data.str['activities'].fillna(v5_processed.str['company_activities']).fillna(v5_company_data.str['activities']).apply(lambda x: x if isinstance(x, list) else []).rename('company_activities'),
    company_data.str['nb_employees'].fillna(v5_company_data.str['number_employees'].astype(float)).rename('company_nb_employees'),
    company_data.str['year_founded'].fillna(v5_company_data.str['year_founded'].astype(float)).rename('company_year_founded'),
    company_data.str['tagline'].fillna(v5_processed.str['tagline']).fillna(v5_company_data.str['tagline']).rename('company_tagline'),
    company_data.str['organization_type'].fillna(_v5_org_type).rename('company_organization_type'),
    company_data.str['latest_funding_investors'].fillna(v5_company_data.str['investors']).rename('company_latest_funding_investors)'),
    company_data.str['latest_funding_type'].fillna(v5_company_data.str['latest_funding_series']).rename('company_latest_funding_type'),
    company_data.str['latest_funding_year'].fillna(v5_company_data.str['latest_funding_year'].astype(float)).rename('company_latest_funding_year'),
    company_data.str['latest_funding_amount'].fillna(v5_company_data.str['latest_funding_amount'].astype(float)).rename('company_latest_funding_amount'),
    company_data.str['stock_exchange'].fillna(v5_company_data.str['stock_exchange']).rename('company_stock_exchange'),
    company_data.str['stock_symbol'].fillna(v5_company_data.str['stock_exchange']).rename('company_stock_symbol'),

    # Extract location arrays
    job.str['_geoloc'].apply(lambda x: x if isinstance(x, list) else []).apply(lambda x_list: [x.get('lon') for x in x_list] if not isinstance(x_list, float) else []).rename('location_longitudes'),
    job.str['_geoloc'].apply(lambda x: x if isinstance(x, list) else []).apply(lambda x_list: [x.get('lat') for x in x_list] if not isinstance(x_list, float) else []).rename('location_latitudes'),
], axis=1).drop(columns='hits').reset_index(drop=True)

In [540]:
hash_set = set(df['requisition_id'])

In [581]:
x_df = df.sort_values(['estimated_publish_date', 'requisition_id']).reset_index(drop=True).drop(columns=['st_mtime', 'st_size', 'is_expired'])
y_df = all_df.query('requisition_id in @hash_set').sort_values(['estimated_publish_date', 'requisition_id']).reset_index(drop=True).drop(columns=['st_mtime', 'st_size', 'is_expired'])

In [570]:
x_df.shape, y_df.shape

((2397, 119), (2397, 119))

In [571]:
x_df.columns[:1]

Index(['requisition_id'], dtype='object')

In [ ]:
# x_df[x_df.columns[118:]].equals(y_df[x_df.columns[118:]])

# x_df[x_df.columns[:9]].equals(y_df[x_df.columns[:9]])
x_df[x_df.columns[118:]].equals(y_df[x_df.columns[118:]])

True

In [610]:
eq_list = []
neq_list = []
for col in x_df.columns:
    if x_df[col].equals(y_df[col]):
        eq_list.append(col)
    else:
        neq_list.append(col)
eq_list = pd.Series(eq_list)
neq_list = pd.Series(neq_list)

In [606]:
len(eq_list), len(neq_list)

(58, 60)

In [628]:
x_df['hiddenFromUsers'] = x_df['hiddenFromUsers'].apply(lambda x: x if isinstance(x, list) else [])
y_df['hiddenFromUsers'] = y_df['hiddenFromUsers'].apply(lambda x: x if isinstance(x, list) else [])

In [646]:
nperc_list = []
for ncol in neq_list:
    if isinstance(x_df[ncol].iloc[0], list):
        nperc_list.append(sum(x_df[ncol].str.len() == y_df[ncol].str.len()))
    # elif isinstance(x_df[ncol].iloc[0], dict):
    #     nperc_list.append(sum(x_df[ncol].str.len() == y_df[ncol].str.len()))
    else:
        nperc_list.append(perc := sum(x_df[ncol] == y_df[ncol]))
    print(f'{ncol}: {perc/len(x_df):.2f}')


description: 0.00
technical_tools: 0.00
licenses_certifications: 0.00
role_activities: 0.00
language_requirements: 0.00
associates_degree_fields: 0.00
bachelors_degree_fields: 0.00
masters_degree_fields: 0.00
doctorate_degree_fields: 0.00
is_high_school_required: 1.00
is_min_management_leadership_yoe_not_mentioned: 1.00
commitment: 1.00
is_workplace_worldwide_ok: 1.00
workplace_cities: 1.00
workplace_counties: 1.00
workplace_states: 1.00
workplace_countries: 1.00
workplace_continents: 1.00
boundless_workplace_states: 1.00
boundless_workplace_countries: 1.00
boundless_workplace_continents: 1.00
number_of_workplace_cities: 1.00
number_of_workplace_counties: 1.00
number_of_workplace_states: 1.00
number_of_workplace_countries: 1.00
number_of_workplace_continents: 1.00
weekend_availability_required: 1.00
holiday_availability_required: 1.00
overtime_required: 1.00
monthly_min_compensation: 0.92
monthly_max_compensation: 0.92
weekly_min_compensation: 0.92
weekly_max_compensation: 0.92
hourly_

In [653]:
df_all = pd.concat([
    df,
    all_df.query('requisition_id not in @hash_set')
]).sort_values(['estimated_publish_date', 'requisition_id'], ascending=[False, True], ignore_index=True)
df_all

,st_mtime,st_size,requisition_id,job_id,board_token,source,apply_url,collapse_key,is_expired,title,...,company_tagline,company_organization_type,company_latest_funding_investors),company_latest_funding_type,company_latest_funding_year,company_latest_funding_amount,company_stock_exchange,company_stock_symbol,location_longitudes,location_latitudes
0,2026-05-30 01:15:46.931586981,4913,x1sbbwkw28y6qvrz,adhoc___meta___2082565018957406,meta,adhoc,https://www.metacareers.com/profile/job_detail...,fb54cc67bc7ac6dc83434a9f2757d7b4aa6a280ee4d859...,False,"Lead, Product Content Engineering",...,Meta develops social networking platforms and ...,Public,None,None,NaN,NaN,NASDAQ,META,"[-122.1491, -74.006, -122.4194]","[37.4542, 40.7128, 37.7749]"
1,2026-05-30 01:22:40.010107756,6778,iohikrbigmby2tgq,governmentjobs___iowa___5359123,iowa,governmentjobs,https://www.governmentjobs.com/careers/iowa/jo...,29b07a1d0db3d9708c6c24fa487578ec7277ded146e5fb...,False,Business Systems Analyst (SCA 111262),...,Provides government administration and public ...,Government,None,None,NaN,NaN,None,None,[-93.617],[41.5825]
2,2026-05-30 01:23:06.212635040,4689,459t9gaqb9umtx4w,successfactors___com___gainwellte___1357001600,com_gainwellte,successfactors,https://jobs.gainwelltechnologies.com/job/Any-...,1122d9601599bf11bf0874ac558479db718e9abd6b6f9b...,False,Senior Data Mining & Payment Integrity Analyst...,...,Delivers technology solutions for public healt...,Private,[Veritas Capital],Private Equity,2020.0,5.000000e+09,None,None,[],[]
3,2026-05-30 01:18:20.974421501,4726,fv9jmij90efjspne,icims2___aarp___7416,aarp,icims2,https://careers.aarp.org/jobs/7416?lang=en-us,5e4e29acf02c11b51245fdc80c66b3ed12e610ebf2f574...,False,"Engineer I, AI Agents",...,Nonprofit advocacy organization serving Americ...,Non-Profit,None,None,NaN,NaN,None,None,[],[]
4,2026-05-30 01:22:41.271778584,5134,p3vqx9j2xl4gx8j2,ashby___talkiatry___c20f10a0-b493-4109-866c-a6...,talkiatry,ashby,https://jobs.ashbyhq.com/talkiatry/c20f10a0-b4...,b41a4402361253056cb20147d1bfad1f86ec7180085a18...,False,Data Engineer,...,None,Private,[Perceptive Advisors],Series D,2026.0,2.100000e+08,None,None,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11109,2026-05-08 22:03:08.651313305,110361,3dp4i78lijqp9dkx,icims___treliantllc___b38ffd47-59cd-46dc-bb8a-...,treliantllc,icims,https://careers-treliant.icims.com/jobs/1803/j...,c4fdab5f4c0db0f98deddc1f7ef85e6e1b012975a7d200...,False,"Consultant, Risk Management",...,Provides compliance and risk management consul...,Private,[Huron Consulting Group],Corporate Round,2025.0,NaN,None,None,[],[]
11110,2026-05-08 22:03:08.651313305,110361,wbf5h3w5tz5zly01,taleo_careersection_s01stantec_230000P8,s01stantec,taleo_careersection,https://s01stantec.taleo.net/careersection/ex1...,cc10f9e7e974fb3278db4d585682047299deea5f615ebb...,False,Utility Economics & Financial Forecasting Analyst,...,Global engineering and architecture firm provi...,Public,None,None,NaN,NaN,New York Stock Exchange,STN,[],[]
11111,2026-05-08 22:03:08.651313305,110361,jxi8mno1sicbn9ks,icims___bottomline___118a1ead-c768-440f-8563-3...,bottomline,icims,https://careers-bottomline.icims.com/jobs/1471...,9d6296e83f0efd2f29f9367d5e63c75152c905ac230c30...,False,Director of Analytics and Reporting,...,Providing college access and graduation suppor...,Non-Profit,None,None,NaN,NaN,None,None,[],[]
11112,2026-05-08 22:03:08.651313305,110361,5l6h3w5mb45sm6wc,breezy___vianai-systems___2709786e7a57,vianai-systems,breezy,https://vianai-systems.breezy.hr/p/2709786e7a5...,6380e5f1d8f422da23a203da1d7a74653ba07a9b256354...,False,Data Scientist,...,Develops artificial intelligence software for ...,Private,[SoftBank Vision Fund 2],Series B,2021.0,1.400000e+08,None,None,[],[]


In [655]:
df_all_parquet_df = _process_df(df_all)
df_all_parquet_df.to_parquet(f'../data/cache/df.parquet')